# 04 - Varianza y boxplot semanal (interactivo, telemetría fija y CropX)

Boxplot por día/semana con bandas de referencia y boxplot semanal estilo publicación, para las dos bases.
Las etapas sin datos se muestran como una figura con el aviso "Sin datos".

> Ejecutar las celdas en orden. Usa `config.py` y `funciones_comunes.py` (misma carpeta).

In [1]:
# =============================================================================
# LIBRERÍAS
# =============================================================================
import re
import warnings
from dataclasses import dataclass
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from matplotlib.gridspec import GridSpec
import seaborn as sns
import ipywidgets as widgets
from scipy import stats
from IPython.display import display, Markdown

warnings.filterwarnings("ignore", category=FutureWarning)
# Gradientes de color sobre filas sin datos válidos (sensor descartado): aviso inofensivo de pandas
warnings.filterwarnings("ignore", message="All-NaN slice encountered")
pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 200)

In [2]:
# =============================================================================
# CONFIGURACION Y FUNCIONES COMUNES (editar config.py y reiniciar el kernel)
# =============================================================================
from config import *
from funciones_comunes import (_limites_rango, obtener_rangos, periodo_finca, filtrar_etapa, columnas_analisis,
                               cargar_finca, cargar_bd_fija, cargar_cropx, cargar_bd_cropx,
                               mostrar_sin_datos, contar_por_banda, _ajustar_slider)

In [3]:
# Telemetría fija y CropX de todas las fincas, recortadas al periodo de sus etapas (incluye Descanso)
BD_FIJA_FILTERED = cargar_bd_fija()
BD_CROPX = cargar_bd_cropx()
BASES = {NOMBRE_BD_FIJA: BD_FIJA_FILTERED, NOMBRE_BD_CROPX: BD_CROPX}


Cargando datos de la finca: La_Orquidea
Se eliminaron 0 filas con valores NaN en la columna 'timestamp' y 0 filas con índice NaN.
Datos cargados para La_Orquidea: 17447 filas y 43 columnas.

Cargando datos de la finca: Venecia
Se eliminaron 0 filas con valores NaN en la columna 'timestamp' y 0 filas con índice NaN.
Datos cargados para Venecia: 17199 filas y 43 columnas.

Cargando datos de la finca: Puerto_Arturo
Se eliminaron 0 filas con valores NaN en la columna 'timestamp' y 0 filas con índice NaN.
Datos cargados para Puerto_Arturo: 17064 filas y 43 columnas.
La_Orquidea: fechas en reloj de 12 h sin AM/PM -> 8733 registros pasados a la tarde según el orden del archivo.
La_Orquidea: 17449 registros CropX, 2025-10-21 10:00:00 a 2026-10-19 22:30:00
Venecia: fechas en reloj de 12 h sin AM/PM -> 8736 registros pasados a la tarde según el orden del archivo.
Venecia: 17450 registros CropX, 2025-10-21 11:00:00 a 2026-10-19 23:30:00
Puerto_Arturo: fechas en reloj de 12 h sin AM/PM -> 3360 re

## Interactive

### Graficos de Varianza y Promedio

In [4]:
# =============================================================================
# GRAFICOS DE VARIANZA (boxplot por periodo + bandas de referencia)
# =============================================================================
from matplotlib import cbook


def graficar_varianza(df: pd.DataFrame, variable1: str, variable2: str | None, titulo: str,
                      bandas: list[tuple[float, float]] | None = None, frecuencia: str = "D") -> None:
    """Boxplot de variable1 por periodo, con variable2 (media por periodo) en eje secundario.

    Args:
        df (pd.DataFrame): Telemetría con índice temporal.
        variable1 (str): Variable de los boxplots (eje izquierdo).
        variable2 (str | None): Variable opcional, se dibuja su media por periodo (eje derecho).
        titulo (str): Título de la figura.
        bandas (list[tuple[float, float]] | None): Bandas (mínimo, máximo) a dibujar como líneas.
        frecuencia (str): Periodo de agrupación de pandas ("D" día, "W" semana).

    Returns:
        None: Muestra la figura.
    """
    periodo = df.index.to_period(frecuencia)

    todos = pd.period_range(periodo.min(), periodo.max(), freq=frecuencia)  # incluye periodos sin datos

    etiquetas = todos.start_time.strftime("%d/%m/%y")
    grupos = df[variable1].groupby(periodo)
    posiciones, estadisticas = [], []

    for i, p in enumerate(todos):                        # periodos sin datos quedan como hueco
        if p in grupos.groups:
            v = grupos.get_group(p).dropna().to_numpy()
            if len(v):
                posiciones.append(i)
                estadisticas.append(cbook.boxplot_stats(v)[0])   # una caja por periodo

    fig, ax1 = plt.subplots(figsize=(15, 6))
    
    ax1.bxp(estadisticas, positions=posiciones, widths=0.6, patch_artist=True,
            boxprops={"facecolor": "skyblue", "linewidth": 0.8},
            medianprops={"color": "black"}, flierprops={"markersize": 2})
    
    ax1.set_xlim(-1, len(todos))
    ax1.set_xticks([])                                   # bxp pone sus propios ticks; se reemplazan abajo
    ax1.set_ylabel(variable1)

    for bmin, bmax in bandas or []:
        ax1.axhline(bmin, color="red", linestyle="--", linewidth=1)
        ax1.axhline(bmax, color="red", linestyle="--", linewidth=1)

    if variable2:
        media2 = df[variable2].groupby(periodo).mean().reindex(todos)
        ax2 = ax1.twinx()
        ax2.plot(range(len(todos)), media2.to_numpy(), color="red", marker=".", linewidth=1.2)
        ax2.set_ylabel(f"{variable2} (media)", color="red")
        ax2.tick_params(axis="y", labelcolor="red")

    paso = max(1, int(np.ceil(len(todos) / 40)))           # máximo ~40 etiquetas en el eje x
    ax1.set_xticks(range(0, len(todos), paso))
    ax1.set_xticklabels(etiquetas[::paso], rotation=90)
    ax1.set_xlabel("Periodo")
    ax1.set_title(titulo, fontsize=13)
    plt.tight_layout()
    plt.show()


def panel_varianza(bd: dict[str, pd.DataFrame], nombre_bd: str, n_bandas: int = 4) -> None:
    """Widget: finca, etapa, variables, agrupación y bandas para el gráfico de varianza.

    Args:
        bd (dict[str, pd.DataFrame]): {finca: datos con índice temporal} (BD_FIJA_FILTERED o BD_CROPX).
        nombre_bd (str): Nombre de la base para los títulos.
        n_bandas (int): Número de sliders de bandas. Una banda con mínimo == máximo no se usa.

    Returns:
        None: Muestra los widgets, la figura y la tabla de conteo por banda.
    """
    if not bd:
        mostrar_sin_datos(f"Varianza - {nombre_bd}", "No hay fincas cargadas en esta base")
        return
    w_finca = widgets.Dropdown(options=list(bd), description="Finca")
    w_etapa = widgets.Dropdown(description="Etapa")
    w_var1 = widgets.Dropdown(description="Variable 1")
    w_var2 = widgets.Dropdown(description="Variable 2")
    w_frec = widgets.Dropdown(options=list(FRECUENCIAS), description="Agrupar")
    w_bandas = [widgets.FloatRangeSlider(description=f"Banda {i + 1}", min=0, max=1, value=(0, 0),
                                         continuous_update=False, readout_format=".2f")
                for i in range(n_bandas)]

    def _al_cambiar_finca(_=None):
        df = bd[w_finca.value]
        variables = list(columnas_analisis(df))
        w_etapa.options = ["Ciclo completo", *obtener_rangos(FINCAS_POR_NOMBRE[w_finca.value])]
        w_etapa.value = "Ciclo completo"
        w_var1.options = variables
        w_var1.value = variables[0] if variables else None
        w_var2.options = [("Ninguna", None), *[(v, v) for v in variables]]
        w_var2.value = None
        _al_cambiar_variable()                           # recalcula sliders aunque la variable no cambie

    def _al_cambiar_variable(_=None):
        if w_var1.value is None:
            return
        serie = bd[w_finca.value][w_var1.value]
        for s in w_bandas:
            _ajustar_slider(s, float(serie.min()), float(serie.max()))

    def _actualizar(finca, etapa, variable1, variable2, frecuencia, **bandas):
        if not variable1 or etapa is None:
            return
        df = filtrar_etapa(bd[finca], finca, None if etapa == "Ciclo completo" else etapa)
        titulo = (f"Distribución de {variable1}" + (f" y {variable2}" if variable2 else "")
                  + f" - {nombre_bd} - {finca} - {etapa}")
        if df[variable1].dropna().empty:
            mostrar_sin_datos(titulo)
            return

        activas = [b for b in bandas.values() if b[0] != b[1]]
        graficar_varianza(df, variable1, variable2, titulo, activas, FRECUENCIAS[frecuencia])
        if activas:
            display(contar_por_banda(df[variable1], activas).style.format(
                {"% Dentro": "{:.2f} %", "% Fuera": "{:.2f} %"}))

    _al_cambiar_finca()
    w_finca.observe(_al_cambiar_finca, "value")
    w_var1.observe(_al_cambiar_variable, "value")

    controles = {"finca": w_finca, "etapa": w_etapa, "variable1": w_var1, "variable2": w_var2,
                 "frecuencia": w_frec, **{f"b{i}": s for i, s in enumerate(w_bandas)}}
    salida = widgets.interactive_output(_actualizar, controles)
    display(widgets.HBox([widgets.VBox([w_finca, w_etapa, w_frec]),
                          widgets.VBox([w_var1, w_var2]),
                          widgets.VBox(w_bandas)]), salida)

In [5]:
panel_varianza(BD_FIJA_FILTERED, NOMBRE_BD_FIJA)

Output()

In [6]:
panel_varianza(BD_CROPX, NOMBRE_BD_CROPX)

Output()

## BOXPLOT

In [7]:
# =============================================================================
# BOXPLOT SEMANAL (estilo publicación) CON VARIABLE SECUNDARIA OPCIONAL
# =============================================================================
from matplotlib import cbook

def cajas_por_periodo(serie: pd.Series, frecuencia: str = "W") -> tuple[pd.PeriodIndex, list[int], list[dict]]:
    """Calcula las estadísticas de boxplot de cada periodo (semana por defecto).

    Args:
        serie (pd.Series): Variable con índice temporal.
        frecuencia (str): Periodo de pandas ("W" = semana de lunes a domingo).

    Returns:
        tuple[pd.PeriodIndex, list[int], list[dict]]: Todos los periodos del rango (incluye los
            vacíos), posición de cada caja con datos y sus estadísticas para `ax.bxp`.
    """
    periodo = serie.index.to_period(frecuencia)
    todos = pd.period_range(periodo.min(), periodo.max(), freq=frecuencia)
    grupos = serie.dropna().groupby(periodo[serie.notna()])
    posiciones, estadisticas = [], []
    for i, p in enumerate(todos):
        if p in grupos.groups:
            posiciones.append(i)
            estadisticas.append(cbook.boxplot_stats(grupos.get_group(p).to_numpy())[0])
    return todos, posiciones, estadisticas


def boxplot_semanal(df: pd.DataFrame, variable1: str, variable2: str | None = None,
                    archivo: str | None = None, max_etiquetas: int = 14) -> None:
    """Boxplot semanal de variable1 y media semanal de variable2 en eje secundario.

    Args:
        df (pd.DataFrame): Telemetría con índice temporal.
        variable1 (str): Variable de los boxplots (eje izquierdo, azul).
        variable2 (str | None): Variable opcional; se dibuja su media semanal (eje derecho, rojo).
        archivo (str | None): Nombre base para guardar SVG y PDF en DIR_SALIDA. None no guarda.
        max_etiquetas (int): Máximo de fechas escritas en el eje x.

    Returns:
        None: Muestra (y opcionalmente guarda) la figura.
    """
    todos, posiciones, estadisticas = cajas_por_periodo(df[variable1], "W")
    color1, color2 = "#1f77b4", "red"

    with plt.rc_context(ESTILO_PUBLICACION), sns.axes_style("white"):
        fig, ax1 = plt.subplots(figsize=(15, 6))
        ax1.bxp(estadisticas, positions=posiciones, widths=0.6, patch_artist=True,
                boxprops={"facecolor": color1, "alpha": 0.8, "linewidth": 0.8},
                medianprops={"color": "black"}, flierprops={"markersize": 2})
        ax1.set_ylabel(variable1, color=color1, fontsize=10)
        ax1.tick_params(axis="y", labelcolor=color1, labelsize=9)

        if variable2:
            semana = df.index.to_period("W")
            media2 = df[variable2].groupby(semana).mean().reindex(todos)
            ax2 = ax1.twinx()
            ax2.plot(range(len(todos)), media2.to_numpy(), color=color2,
                     linestyle="--", linewidth=1.8, marker="o", markersize=3)
            ax2.set_ylabel(f"{variable2} (media semanal)", color=color2, fontsize=10)
            ax2.tick_params(axis="y", labelcolor=color2, labelsize=9)

        # Eje x: una marca por semana, etiqueta (lunes de la semana) solo cada `paso` semanas
        paso = max(1, int(np.ceil(len(todos) / max_etiquetas)))
        ax1.set_xlim(-0.7, len(todos) - 0.3)
        ax1.set_xticks(range(len(todos)), minor=True)
        ax1.set_xticks(range(0, len(todos), paso))
        ax1.set_xticklabels(todos.start_time.strftime("%d/%m/%y")[::paso],
                            rotation=45, ha="right", rotation_mode="anchor", fontsize=9)
        ax1.set_xlabel("Semana (fecha del lunes)", fontsize=10)

        fig.tight_layout()
        if archivo:
            DIR_SALIDA.mkdir(parents=True, exist_ok=True)
            for ext in ("svg", "pdf"):
                fig.savefig(DIR_SALIDA / f"{archivo}.{ext}", format=ext, bbox_inches="tight")
            print(f"Guardado en: {DIR_SALIDA / archivo}.svg / .pdf")
        plt.show()


def panel_boxplot_semanal(bd: dict[str, pd.DataFrame], nombre_bd: str) -> None:
    """Widget: finca, etapa, variables y guardado opcional del boxplot semanal.

    Args:
        bd (dict[str, pd.DataFrame]): {finca: datos con índice temporal} (BD_FIJA_FILTERED o BD_CROPX).
        nombre_bd (str): Nombre de la base para los mensajes y el nombre del archivo.

    Returns:
        None: Muestra los widgets y la figura.
    """
    if not bd:
        mostrar_sin_datos(f"Boxplot semanal - {nombre_bd}", "No hay fincas cargadas en esta base")
        return
    w_finca = widgets.Dropdown(options=list(bd), description="Finca")
    w_etapa = widgets.Dropdown(description="Etapa")
    w_var1 = widgets.Dropdown(description="Variable 1")
    w_var2 = widgets.Dropdown(description="Variable 2")
    w_guardar = widgets.Checkbox(value=False, description="Guardar SVG/PDF")
    w_archivo = widgets.Text(value=f"boxplot_semanal_{nombre_bd}", description="Archivo")

    def _al_cambiar_finca(_=None):
        variables = list(columnas_analisis(bd[w_finca.value]))
        w_etapa.options = ["Ciclo completo", *obtener_rangos(FINCAS_POR_NOMBRE[w_finca.value])]
        w_etapa.value = "Ciclo completo"
        w_var1.options = variables
        w_var1.value = variables[0] if variables else None
        w_var2.options = [("Ninguna", None), *[(v, v) for v in variables]]
        w_var2.value = None

    def _actualizar(finca, etapa, variable1, variable2, guardar, archivo):
        if not variable1 or etapa is None:
            return
        df = filtrar_etapa(bd[finca], finca, None if etapa == "Ciclo completo" else etapa)
        if df[variable1].dropna().empty:
            mostrar_sin_datos(f"{variable1} - {nombre_bd} - {finca} - {etapa}")
            return
        print(f"{nombre_bd} - {finca} - {etapa} - {df.index.min():%d/%m/%Y} a {df.index.max():%d/%m/%Y}")
        boxplot_semanal(df, variable1, variable2, archivo if guardar and archivo else None)

    _al_cambiar_finca()
    w_finca.observe(_al_cambiar_finca, "value")

    salida = widgets.interactive_output(_actualizar, {
        "finca": w_finca, "etapa": w_etapa, "variable1": w_var1,
        "variable2": w_var2, "guardar": w_guardar, "archivo": w_archivo})
    display(widgets.HBox([widgets.VBox([w_finca, w_etapa]),
                          widgets.VBox([w_var1, w_var2]),
                          widgets.VBox([w_guardar, w_archivo])]), salida)

In [8]:
panel_boxplot_semanal(BD_FIJA_FILTERED, NOMBRE_BD_FIJA)

Output()

In [9]:
panel_boxplot_semanal(BD_CROPX, NOMBRE_BD_CROPX)

Output()